# Visualize evaluation results

In [ ]:
from matplotlib import pyplot as plt
from matplotlib.patches import Polygon
import numpy as np

import config
import evaluator

## Read results from files

In [ ]:
results = {}
for scope in config.SCOPES:
    results[scope] = {}
    for model_name in config.MODELS:
        results[scope][model_name] = {}
        for method_name in config.METHODS[scope]:
            if method_name in ['ra', 'g', 'gxi']:
                df = evaluator.read_results_from_file(model_name,
                                                      scope,
                                                      method_name)
                if df is not None:
                    results[scope][model_name][method_name] = df
            elif method_name == 'ig':
                for n_steps in config.N_STEPS:
                    method_name_new = method_name + str(n_steps)
                    df = evaluator.read_results_from_file(model_name,
                                                          scope,
                                                          method_name,
                                                          n_steps=n_steps)
                    if df is not None:
                        results[scope][model_name][method_name_new] = df
            elif method_name == 'shap':
                for n_samples in config.N_SAMPLES:
                    method_name_new = method_name + str(n_samples)
                    df = evaluator.read_results_from_file(model_name,
                                                          scope,
                                                          method_name,
                                                          n_samples=n_samples)
                    if df is not None:
                        results[scope][model_name][method_name_new] = df

## Plot results

In [ ]:
box_colors = ['darkseagreen',
              'palevioletred',
              'khaki',
              'steelblue',
              'mediumpurple',
              'paleturquoise',
              'burlywood',
              'cornflowerblue',
              'plum',
              'olivedrab']

In [ ]:
from statistics import median
medians = {}
for metric in ['infid', 'time']:
    medians[metric] = {}
    for scope in config.SCOPES:
        medians[metric][scope] = {}
        fig, axs = plt.subplots(1, 2, figsize=(12,4))
        for i, model_name in enumerate(results[scope].keys()):
            method_names = list(results[scope][model_name].keys())
            values = [results[scope][model_name][method_name][metric].values for method_name in method_names]
            medians[metric][scope][model_name] = {method_name: round(median(values[i]),4) for (method_name,i) in zip(method_names, range(len(values)))}
            # Horizontal plot without outlier points
            bp = axs[i].boxplot(values, sym='', vert=False)
            plt.setp(bp['boxes'], color='black')
            plt.setp(bp['whiskers'], color='black')
            # Add vertical grid and hide it behind plot objects
            axs[i].xaxis.grid(True, linestyle='-', which='major', color='lightgrey', alpha=0.5)
            if metric == 'time':
                xlabel = metric + ' in s'
            else:
                xlabel = metric
            axs[i].set(title=model_name.upper(), axisbelow=True, xlabel=xlabel+' ('+scope+')')
            # Fill boxes with desired colors
            num_boxes = len(values)
            for j in range(num_boxes):
                box = bp['boxes'][j]
                box_x = []
                box_y = []
                length = len(box.get_xdata())
                for k in range(length):
                    box_x.append(box.get_xdata()[k])
                    box_y.append(box.get_ydata()[k])
                box_coords = np.column_stack([box_x, box_y])
                axs[i].add_patch(Polygon(box_coords, facecolor=box_colors[j]))
            # Display method names
            axs[i].set_yticklabels([m.upper() for m in method_names], rotation=0, fontsize=10)
            # Save plot
            basename = f"plots/boxplot_{metric}_{scope}"
            fig.savefig(f"{basename}.eps", format="eps")
            fig.savefig(f"{basename}.png", format="png")


In [ ]:
print(medians['infid']['local']['laat'])